# 2. Training (GPU T4 x2): English on GPU 0, Telugu on GPU 1 in parallel

Kaggle settings: **Accelerator: GPU T4 x2**, **Internet: On**.
**Add Input -> Your Work ->** the data notebook from step 1 (its output).
To resume a previous training run, also add that training notebook's output.

Run with **Save Version -> Save & Run All (Commit)**: it runs up to 12 h headless and
saves checkpoints and logs in the Output. `MAX_HOURS` stops training cleanly before
Kaggle's 12 h limit.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/snehithareddy28/fastspeech2-assignment.git"
os.chdir("/kaggle/working")
if not os.path.exists("fastspeech2-assignment"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/kaggle/working/fastspeech2-assignment")
!git pull
!pip install -q -r requirements.txt

In [ ]:
import glob, shutil
# Link the prepared features from the data notebook output
for lang in ["English", "Telugu"]:
    src = glob.glob(f"/kaggle/input/*/fastspeech2-assignment/data/{lang}/preprocessed")
    assert src, f"attach the data notebook output ({lang} preprocessed not found)"
    os.makedirs(f"data/{lang}", exist_ok=True)
    if not os.path.exists(f"data/{lang}/preprocessed"):
        os.symlink(src[0], f"data/{lang}/preprocessed")
    for f in glob.glob(f"/kaggle/input/*/fastspeech2-assignment/Results/{lang}/*.json") + \
             glob.glob(f"/kaggle/input/*/fastspeech2-assignment/Results/{lang}/*.t*"):
        shutil.copy(f, f"Results/{lang}/")
    # resume: copy checkpoints and logs from a previous training run, if attached
    prev = sorted(glob.glob(f"/kaggle/input/*/fastspeech2-assignment/{lang}/checkpoints/latest.pth.tar"))
    if prev and not os.path.exists(f"{lang}/checkpoints/latest.pth.tar"):
        d = os.path.dirname(prev[-1])
        for f in glob.glob(d + "/*.pth.tar"):
            shutil.copy(f, f"{lang}/checkpoints/")
        shutil.copytree(d.replace(f"{lang}/checkpoints", f"Results/{lang}/logs"),
                        f"Results/{lang}/logs", dirs_exist_ok=True)
        print(lang, "resuming from", prev[-1])
!nvidia-smi -L

In [ ]:
MAX_HOURS = 11
!bash scripts/run_pipeline.sh train-both {MAX_HOURS}
!tail -3 Results/English/logs/train_stdout.log Results/Telugu/logs/train_stdout.log

In [ ]:
# Results: loss plots, fp16 export, test-sentence audio + variance plots
!bash scripts/run_pipeline.sh results english
!bash scripts/run_pipeline.sh results telugu
# keep the output small: drop numbered checkpoints, keep latest + best
!rm -f English/checkpoints/[0-9]*.pth.tar Telugu/checkpoints/[0-9]*.pth.tar
!du -sh English/checkpoints Telugu/checkpoints Results